# 15. Report tables and figures

Builds every table the 5,000-word report needs, in final numbered form, and writes them to
`writing/report_tables.md` as Markdown you can paste straight into Word.

**No GPU. No training. Runs in a few seconds on a laptop.** It reads the result CSVs that
already exist and reformats them. Nothing here recomputes a model.

The point of this notebook is to make the Results chapter assembly rather than computation.
Once it has run, writing that chapter is a matter of putting sentences between tables that
already exist and are already correct.

Every section degrades gracefully: if a file is missing it says so and carries on, so this
runs before or after notebook 14.

| Table | Content | Source |
|---|---|---|
| 1 | Corpus composition | `loaders` |
| 2 | Feature ablation, baseline | `cv_ablation.csv` |
| 3 | Within-dataset model comparison | `all_results.csv` |
| 4 | Sensitivity to change of seed | `multiseed_results.csv` |
| 5 | Cross-dataset transfer | `multiseed_results.csv`, `all_results.csv` |
| 6 | Adversarial attack | `multiseed_results.csv`, `attack_curve_baseline.csv` |
| 7 | Confusion matrix counts | `confusion_counts.csv` (notebook 14) |
| 8 | Confidence-weighted score | `confidence_scores.csv` (notebook 14) |
| 9 | Error analysis categories | hand-categorised CSV |

In [1]:
import os, re
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = Path(os.environ.get('HINGLISH_HATE_ROOT', r'H:\Data\msc-hinglish-hate'))
if not (ROOT / 'data').exists():
    ROOT = Path.cwd().parent
import sys; sys.path.insert(0, str(ROOT / 'notebooks'))

DATA = ROOT / 'data'
W    = ROOT / 'writing'
FIG  = W / 'figures'; FIG.mkdir(parents=True, exist_ok=True)
print('root:', ROOT)

OUT = []          # accumulates the Markdown
def section(title):
    OUT.append(f'\n\n## {title}\n')
    print(f'\n--- {title} ---')

def emit(df, caption=''):
    """Print a dataframe and append it to the Markdown output."""
    if caption:
        OUT.append(f'\n*{caption}*\n')
    OUT.append('\n' + df.to_markdown(index=False) + '\n')
    print(df.to_string(index=False))

def note(msg):
    OUT.append(f'\n> {msg}\n')
    print('  !', msg)

def read(name):
    p = W / name
    if not p.exists():
        return None
    return pd.read_csv(p)

root: H:\Data\msc-hinglish-hate


## Table 1. Corpus composition

The three source corpora, before and after the romanised filter. This is the table that
justifies why the frozen split is built from Bohra alone and HASOC 2022 is held out entirely.

In [2]:
section('Table 1. Corpus composition')
try:
    from hinglish_hate import (load_bohra, load_hasoc2021, load_hasoc2022_threads,
                               filter_romanised)
    # load_bohra takes the .tsv itself; the HASOC loaders walk a directory tree
    rows = []
    for name, loader, arg in [('Bohra et al. (2018)', load_bohra, DATA / 'hate_speech.tsv'),
                              ('HASOC 2021 ICHCL',    load_hasoc2021,        DATA),
                              ('HASOC 2022 ICHCL',    load_hasoc2022_threads, DATA)]:
        d = loader(arg)
        r = filter_romanised(d, include_mixed=True)
        rows.append({'Corpus': name,
                     'Posts': len(d),
                     'Hate %': round(100 * d['label'].mean(), 1),
                     'Romanised': len(r),
                     'Romanised %': round(100 * len(r) / max(len(d), 1), 1)})
    t1 = pd.DataFrame(rows)
    t1.loc[len(t1)] = {'Corpus': 'Total', 'Posts': t1['Posts'].sum(), 'Hate %': np.nan,
                       'Romanised': t1['Romanised'].sum(), 'Romanised %': np.nan}
    emit(t1, 'Table 1. Source corpora before and after the romanised filter.')
except Exception as e:
    note(f'Could not rebuild from loaders ({type(e).__name__}). Fill Table 1 by hand.')


--- Table 1. Corpus composition ---
[load_bohra] 4574 rows | fixed 0 label typo(s) | dropped 4 unparseable
             Corpus  Posts  Hate %  Romanised  Romanised %
Bohra et al. (2018)   4574    36.3       4574        100.0
   HASOC 2021 ICHCL   4911    47.8       4335         88.3
   HASOC 2022 ICHCL   4901    51.4       4306         87.9
              Total  14386     NaN      13215          NaN


## Table 2. Feature ablation

What the character n-grams and the lexicon features actually buy over word n-grams alone.
This is the table that justifies the feature union rather than a plain TF-IDF baseline, and
it is five-fold cross-validated so the comparison is not a single-split artefact.

In [3]:
section('Table 2. Feature ablation, baseline')
d = read('cv_ablation.csv')
if d is None:
    note('cv_ablation.csv not found.')
else:
    t2 = d.rename(columns={'features': 'Feature set',
                           'macro_f1_mean': 'Macro-F1',
                           'macro_f1_std': 'SD',
                           'accuracy': 'Accuracy',
                           'precision': 'Precision',
                           'recall': 'Recall'})
    num = t2.select_dtypes('number').columns
    t2[num] = t2[num].round(3)
    emit(t2, 'Table 2. Feature ablation, five-fold cross-validation on the training split.')


--- Table 2. Feature ablation, baseline ---
      Feature set  Macro-F1    SD  Accuracy  Precision  Recall
 word-only TF-IDF     0.622 0.010     0.644      0.621   0.626
word+char+lexicon     0.657 0.005     0.680      0.656   0.658


## Table 3. Within-dataset comparison

Every model at its best configuration on the frozen 915-post test set. Degenerate runs are
excluded from the table but reported underneath, because a run that collapsed to a single
class is a finding about fine-tuning stability rather than a result to hide.

In [4]:
section('Table 3. Within-dataset comparison')
d = read('all_results.csv')
if d is None:
    note('all_results.csv not found.')
else:
    keep = d[d['degenerate'].astype(str).str.lower().isin(['no', 'false', 'nan'])].copy()
    best = (keep.sort_values('within_macro_f1', ascending=False)
                .groupby('model', as_index=False).first())
    cols = ['model', 'protocol', 'epochs', 'lr', 'within_macro_f1', 'within_hate_f1',
            'within_f1_not', 'accuracy']
    t3 = best[[c for c in cols if c in best.columns]].copy()
    t3 = t3.sort_values('within_macro_f1', ascending=False)
    t3.columns = ['Model', 'Protocol', 'Epochs', 'LR', 'Macro-F1', 'Hate F1',
                  'Not-hate F1', 'Accuracy'][:len(t3.columns)]
    num = t3.select_dtypes('number').columns
    t3[num] = t3[num].round(3)
    emit(t3, 'Table 3. Best configuration per model, frozen 915-post test set.')

    deg = d[d['degenerate'].astype(str).str.lower().isin(['yes', 'true'])]
    if len(deg):
        OUT.append('\n**Degenerate runs, reported for completeness:**\n')
        dd = deg[['model', 'epochs', 'lr', 'within_macro_f1', 'within_hate_f1', 'note']].copy()
        dd[['within_macro_f1', 'within_hate_f1']] = dd[['within_macro_f1', 'within_hate_f1']].round(3)
        emit(dd, 'Runs that collapsed to a single predicted class.')


--- Table 3. Within-dataset comparison ---
                            Model     Protocol  Epochs  LR  Macro-F1  Hate F1  Not-hate F1  Accuracy
                 xlm-roberta-base single split     3.0 0.0     0.665    0.565        0.764     0.694
          google/muril-base-cased single split     6.0 0.0     0.665    0.543        0.787     0.709
TF-IDF word+char+lexicon + LogReg    5-fold CV     NaN NaN     0.657    0.567        0.729     0.680
             ai4bharat/indic-bert single split     6.0 0.0     0.640    0.537        0.742     0.669
        TF-IDF word-only + LogReg    5-fold CV     NaN NaN     0.622      NaN          NaN       NaN
           model  epochs      lr  within_macro_f1  within_hate_f1                                                                                                                                                 note
xlm-roberta-base     6.0 0.00003            0.312           0.531 DEGENERATE: loss flat at ln(2)=0.693; predicted hate for 874/915. Cro

## Table 4. Sensitivity to change of seed

Hunter's phrasing, not "seed variance". His objection on 29 September was that variance has a
specific statistical meaning and should not be borrowed loosely.

This is the table that retires the claim made at the formative viva that the transformers beat
the baseline. Once each model is reported as a mean and a spread over three seeds, the
difference sits inside the spread.

In [5]:
section('Table 4. Sensitivity to change of seed')
d = read('multiseed_results.csv')
if d is None:
    note('multiseed_results.csv not found.')
else:
    g = (d.groupby('model')
           .agg(Seeds=('macro_f1', 'size'),
                within_mean=('macro_f1', 'mean'),   within_sd=('macro_f1', 'std'),
                hate_mean=('hate_f1', 'mean'),      hate_sd=('hate_f1', 'std'),
                cross_mean=('cross_macro_f1', 'mean'), cross_sd=('cross_macro_f1', 'std'))
           .round(3).reset_index())
    t4 = pd.DataFrame({
        'Model': g['model'],
        'Seeds': g['Seeds'],
        'Within-dataset macro-F1': [f'{m:.3f} ± {s:.3f}' for m, s in zip(g['within_mean'], g['within_sd'])],
        'Hate F1':                 [f'{m:.3f} ± {s:.3f}' for m, s in zip(g['hate_mean'], g['hate_sd'])],
        'Cross-dataset macro-F1':  [f'{m:.3f} ± {s:.3f}' for m, s in zip(g['cross_mean'], g['cross_sd'])],
    })
    emit(t4, 'Table 4. Mean and standard deviation over three seeds (42, 1337, 2024).')
    note('Baseline for comparison: within 0.657 ± 0.005 (five-fold CV), cross 0.510.')

    spread = (g['within_mean'].max() - g['within_mean'].min())
    worst_sd = g['within_sd'].max()
    print(f'\n  spread between model means: {spread:.3f}')
    print(f'  largest single-model SD:    {worst_sd:.3f}')
    print('  -> the between-model gap is inside the within-model spread'
          if spread < worst_sd else
          '  -> the between-model gap exceeds the within-model spread')


--- Table 4. Sensitivity to change of seed ---
                  Model  Seeds Within-dataset macro-F1       Hate F1 Cross-dataset macro-F1
   ai4bharat/indic-bert      3           0.656 ± 0.015 0.550 ± 0.004          0.411 ± 0.036
google/muril-base-cased      3           0.660 ± 0.010 0.537 ± 0.025          0.378 ± 0.032
       xlm-roberta-base      3           0.654 ± 0.010 0.525 ± 0.031          0.410 ± 0.057
  ! Baseline for comparison: within 0.657 ± 0.005 (five-fold CV), cross 0.510.

  spread between model means: 0.006
  largest single-model SD:    0.015
  -> the between-model gap is inside the within-model spread


## Table 5. Cross-dataset transfer

The finding that survives the seed analysis. The gap here is several times the seed spread,
so unlike the within-dataset comparison it is not noise.

In [6]:
section('Table 5. Cross-dataset transfer')
d = read('multiseed_results.csv')
if d is None:
    note('multiseed_results.csv not found.')
else:
    g = d.groupby('model').agg(within=('macro_f1', 'mean'),
                               cross=('cross_macro_f1', 'mean'),
                               cross_sd=('cross_macro_f1', 'std')).reset_index()
    g['drop'] = g['within'] - g['cross']
    g['drop_pct'] = 100 * g['drop'] / g['within']
    t5 = pd.DataFrame({
        'Model': g['model'],
        'Within': g['within'].round(3),
        'Cross (HASOC 2022)': [f'{m:.3f} ± {s:.3f}' for m, s in zip(g['cross'], g['cross_sd'])],
        'Absolute drop': g['drop'].round(3),
        'Relative drop %': g['drop_pct'].round(1),
    })
    base = pd.DataFrame([{'Model': 'TF-IDF + LogReg', 'Within': 0.657,
                          'Cross (HASOC 2022)': '0.510', 'Absolute drop': 0.147,
                          'Relative drop %': 22.4}])
    emit(pd.concat([base, t5], ignore_index=True),
         'Table 5. Transfer to a corpus never seen in training.')
    note('The baseline transfers best. Every transformer transfers worse than the '
         'statistical model, by a margin far larger than the seed spread in Table 4.')


--- Table 5. Cross-dataset transfer ---
                  Model  Within Cross (HASOC 2022)  Absolute drop  Relative drop %
        TF-IDF + LogReg   0.657              0.510          0.147             22.4
   ai4bharat/indic-bert   0.656      0.411 ± 0.036          0.245             37.4
google/muril-base-cased   0.660      0.378 ± 0.032          0.282             42.8
       xlm-roberta-base   0.654      0.410 ± 0.057          0.243             37.2
  ! The baseline transfers best. Every transformer transfers worse than the statistical model, by a margin far larger than the seed spread in Table 4.


## Table 6. Adversarial attack

Relative macro-F1 drop at full perturbation, with the realised token change rate alongside,
because the requested rate and the realised rate differ: not every token has a romanisation
variant available.

In [7]:
section('Table 6. Adversarial attack')
d = read('multiseed_results.csv')
if d is None:
    note('multiseed_results.csv not found.')
else:
    rows = []
    for model, grp in d.groupby('model'):
        clean = grp['macro_f1'].mean()
        r = {'Model': model, 'Clean macro-F1': round(clean, 3)}
        for atk, label in [('romanisation', 'Romanisation'), ('character', 'Character')]:
            col = f'{atk}_1.0_macro_f1'
            if col in grp.columns:
                v = grp[col].mean()
                r[f'{label} macro-F1'] = round(v, 3)
                r[f'{label} drop %']   = round(100 * (clean - v) / clean, 1)
                rcol = f'{atk}_1.0_realised'
                if rcol in grp.columns:
                    r[f'{label} realised'] = round(grp[rcol].mean(), 3)
        rows.append(r)
    t6 = pd.DataFrame(rows)

    b = read('attack_curve_baseline.csv')
    if b is not None:
        br = {'Model': 'TF-IDF + LogReg'}
        clean = b[b['requested_rate'] == 0.0]['macro_f1'].mean()
        br['Clean macro-F1'] = round(clean, 3)
        for atk, label in [('romanisation', 'Romanisation'), ('character', 'Character')]:
            sub = b[(b['attack'] == atk) & (b['requested_rate'] == 1.0)]
            if len(sub):
                v = sub['macro_f1'].iloc[0]
                br[f'{label} macro-F1'] = round(v, 3)
                br[f'{label} drop %']   = round(100 * (clean - v) / clean, 1)
                br[f'{label} realised'] = round(sub['realised_rate'].iloc[0], 3)
        t6 = pd.concat([pd.DataFrame([br]), t6], ignore_index=True)

    emit(t6, 'Table 6. Performance under attack at full perturbation rate.')
    note('The transformers are markedly less affected by character corruption than the '
         'baseline. Subword tokenisation re-segments a damaged word into valid pieces and '
         'context recovers the meaning; the TF-IDF features have no context to fall back on.')


--- Table 6. Adversarial attack ---
                  Model  Clean macro-F1  Romanisation macro-F1  Romanisation drop %  Romanisation realised  Character macro-F1  Character drop %  Character realised
        TF-IDF + LogReg           0.642                  0.594                  7.5                  0.762               0.480              25.2                0.71
   ai4bharat/indic-bert           0.656                  0.589                 10.3                  0.762               0.570              13.1                0.71
google/muril-base-cased           0.660                  0.607                  7.9                  0.762               0.544              17.5                0.71
       xlm-roberta-base           0.654                  0.641                  1.9                  0.762               0.632               3.2                0.71
  ! The transformers are markedly less affected by character corruption than the baseline. Subword tokenisation re-segments a damaged word

## Tables 7 and 8. Confusion matrices and the confidence score

These two come from notebook 14. If it has not been run yet, this section says so.

In [8]:
section('Table 7. Confusion matrix counts')
d = read('confusion_counts.csv')
if d is None:
    note('confusion_counts.csv not found. Run notebook 14 first.')
else:
    t7 = d.rename(columns={'model': 'Model', 'split': 'Test set',
                           'true_not': 'TN', 'false_hate': 'FP',
                           'missed_hate': 'FN', 'true_hate': 'TP',
                           'hate_recall': 'Hate recall',
                           'hate_precision': 'Hate precision',
                           'not_recall': 'Not-hate recall'})
    emit(t7, 'Table 7. Confusion matrix cells, both test sets.')
    w = t7[t7['Test set'] == 'within']
    if len(w):
        gap = (w['Not-hate recall'] - w['Hate recall']).mean()
        note(f'Mean recall gap, not-hate minus hate, within-dataset: {gap:+.3f}. '
             'A positive gap is the systematic pattern Hunter was looking for: '
             'the loss is concentrated on the minority class.')

section('Table 8. Confidence-weighted score')
d = read('confidence_scores.csv')
if d is None:
    note('confidence_scores.csv not found. Run notebook 14 first.')
else:
    t8 = d.rename(columns={'model': 'Model', 'split': 'Test set',
                           'confidence_score': 'Confidence score',
                           'mean_D_correct': 'Mean D, correct',
                           'mean_D_wrong': 'Mean D, wrong',
                           'pct_errors_confident': '% errors confident'})
    cols = ['Model', 'Test set', 'accuracy', 'Confidence score',
            'Mean D, correct', 'Mean D, wrong', '% errors confident']
    emit(t8[[c for c in cols if c in t8.columns]],
         'Table 8. Confidence-weighted score after Hunter (2026), with its decomposition.')
    w = t8[t8['Test set'] == 'within']
    if len(w):
        for _, r in w.iterrows():
            verdict = ('wrong but unsure' if r['Mean D, wrong'] < r['Mean D, correct'] - 0.1
                       else 'wrong and confident')
            print(f"  {r['Model']:20s} score {r['Confidence score']:+.3f}  ({verdict})")


--- Table 7. Confusion matrix counts ---


          Model Test set   TN  FP   FN  TP  Hate recall  Hate precision  Not-hate recall
TF-IDF + LogReg   within  415 168  140 192        0.578           0.533            0.712
    XLM-RoBERTa   within  448 135  155 177        0.533           0.567            0.768
          MuRIL   within  479 104  174 158        0.476           0.603            0.822
   IndicBERT v1   within  446 137  154 178        0.536           0.565            0.765
TF-IDF + LogReg    cross 1817 282 1865 342        0.155           0.548            0.866
    XLM-RoBERTa    cross 1632 467 1669 538        0.244           0.535            0.778
          MuRIL    cross 2068  31 2118  89        0.040           0.742            0.985
   IndicBERT v1    cross 2040  59 2102 105        0.048           0.640            0.972
  ! Mean recall gap, not-hate minus hate, within-dataset: +0.236. A positive gap is the systematic pattern Hunter was looking for: the loss is concentrated on the minority class.

--- Table 8. Confid

## Table 9. Error analysis

The hand-categorised breakdown of the missed true-hate posts. This is the table the argument
rests on, and it is the only result in the project that could not have been produced without
reading the posts.

In [9]:
section('Table 9. Error analysis of missed hate posts')
cand = [W / 'true_hate_MISSED_categorised.csv', W / 'true_hate_for_reading.csv']
src = next((p for p in cand if p.exists()), None)
LABELS = {
    'A': 'Implicit group hate, no explicit slur',
    'B': 'Slur present but absent from the lexicon',
    'C': 'Political criticism labelled as hate',
    'D': 'Too short to classify',
    'E': 'Would not be called hate speech on reading',
    'F': 'Ambiguous without further context',
    'G': 'Direct threat',
    'H': 'Personal abuse, no group target and no slur',
}
d = pd.read_csv(src) if src is not None else None
col = next((c for c in (d.columns if d is not None else [])
            if c.lower() in ('code', 'category', 'cat')), None)

if d is not None and col:
    counts = d[col].astype(str).str.strip().str.upper().value_counts()
    t9 = pd.DataFrame({'Code': counts.index,
                       'Category': [LABELS.get(c, '') for c in counts.index],
                       'Posts': counts.values})
    t9['% of misses'] = (100 * t9['Posts'] / t9['Posts'].sum()).round(1)
    emit(t9, f'Table 9. Categories of missed true-hate posts (n={int(t9["Posts"].sum())}).')
    ce = t9[t9['Code'].isin(['C', 'E'])]['% of misses'].sum()
    note(f'Categories C and E together account for {ce:.1f}% of all missed hate posts. '
         'In both the model is marked wrong against a label that is arguable. This is a '
         'ceiling set by the label definition, not by model capacity.')
else:
    note('Categorised file not found in writing/. **Commit '
         '`true_hate_MISSED_categorised.csv` to the repository** — it is the only artefact '
         'of the hand annotation and it is currently untracked. Table 9 filled from the '
         'recorded counts in the meantime.')
    t9 = pd.DataFrame({
        'Code': list('CEABFHDG'),
        'Category': [LABELS[c] for c in 'CEABFHDG'],
        'Posts': [58, 26, 19, 10, 9, 8, 5, 5],
    })
    t9['% of misses'] = (100 * t9['Posts'] / 140).round(1)
    emit(t9, 'Table 9. Categories of missed true-hate posts (n=140).')

dg = read('error_analysis_diagnostics.csv')
if dg is not None:
    emit(dg.round(2), 'Table 9b. Lexicon coverage on caught against missed posts.')


--- Table 9. Error analysis of missed hate posts ---
  ! Categorised file not found in writing/. **Commit `true_hate_MISSED_categorised.csv` to the repository** — it is the only artefact of the hand annotation and it is currently untracked. Table 9 filled from the recorded counts in the meantime.
Code                                    Category  Posts  % of misses
   C        Political criticism labelled as hate     58         41.4
   E  Would not be called hate speech on reading     26         18.6
   A       Implicit group hate, no explicit slur     19         13.6
   B    Slur present but absent from the lexicon     10          7.1
   F           Ambiguous without further context      9          6.4
   H Personal abuse, no group target and no slur      8          5.7
   D                       Too short to classify      5          3.6
   G                               Direct threat      5          3.6
 group   n  mean_words  mean_words_raw  mean_chars  mean_lexicon_hits  mean_lexi

## The summary figure

One figure carrying the central claim: no architecture wins on all three axes. Use it as the
opening figure of the Results chapter, because it tells the reader what the chapter is for.

In [10]:
section('Summary figure: the three-way trade-off')
d = read('multiseed_results.csv')
if d is None:
    note('multiseed_results.csv not found.')
else:
    g = d.groupby('model').agg(within=('macro_f1', 'mean'),
                               cross=('cross_macro_f1', 'mean')).reset_index()
    rob = []
    for m in g['model']:
        grp = d[d['model'] == m]
        clean = grp['macro_f1'].mean()
        worst = min(grp[f'{a}_1.0_macro_f1'].mean()
                    for a in ('romanisation', 'character')
                    if f'{a}_1.0_macro_f1' in grp.columns)
        rob.append(1 - (clean - worst) / clean)      # retained fraction, worst case
    g['retained'] = rob

    names  = ['TF-IDF + LogReg'] + list(g['model'])
    within = [0.657] + list(g['within'])
    cross  = [0.510] + list(g['cross'])
    retain = [1 - 0.252] + list(g['retained'])

    x = np.arange(len(names)); wdt = 0.26
    fig, ax = plt.subplots(figsize=(7.6, 3.8))
    ax.bar(x - wdt, within, wdt, label='Within-dataset macro-F1',  color='#2b7bba')
    ax.bar(x,       cross,  wdt, label='Cross-dataset macro-F1',   color='#7cb5d8')
    ax.bar(x + wdt, retain, wdt, label='Fraction retained, worst attack', color='#d1495b')
    ax.set_xticks(x)
    ax.set_xticklabels([n.replace('/', '/\n') for n in names], fontsize=8)
    ax.set_ylim(0, 1.0); ax.set_ylabel('score')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_title('No architecture wins on all three axes', fontsize=11)
    fig.tight_layout()
    fig.savefig(FIG / 'fig_tradeoff.png', dpi=200, bbox_inches='tight'); plt.close(fig)
    print('saved fig_tradeoff.png')
    OUT.append('\n![Three-way trade-off](figures/fig_tradeoff.png)\n')


--- Summary figure: the three-way trade-off ---
saved fig_tradeoff.png


In [11]:
# -------------------------------------------------------------- write the file
header = (
    '# Report tables\n\n'
    'Generated by `notebooks/15_report_tables.ipynb`. Do not edit by hand: '
    're-run the notebook instead, so the numbers in the report can never drift '
    'from the numbers in the result files.\n'
)
path = W / 'report_tables.md'
path.write_text(header + ''.join(OUT), encoding='utf-8')
print(f'\nwrote {path} ({path.stat().st_size} bytes)')
print('\nPaste these into the report. In Word: Insert > Table > Convert Text to Table, '
      'or paste the Markdown into any converter.')


wrote H:\Data\msc-hinglish-hate\writing\report_tables.md (12184 bytes)

Paste these into the report. In Word: Insert > Table > Convert Text to Table, or paste the Markdown into any converter.


## Using this

Open `writing/report_tables.md`. Every table is numbered in the order it should appear in the
Results chapter, with a caption already written.

The chapter then writes itself in roughly this order, and at 1,300 words it is about 150 words
of prose per table:

1. **Table 3** establishes that every model lands between 0.64 and 0.67 within-dataset.
2. **Table 4** shows that the apparent gap between them is inside the seed spread, so the
   models tie. Say this plainly rather than burying it.
3. **Table 5** shows the baseline transfers best, and that this gap is real because it is
   several times the spread in Table 4.
4. **Table 6** shows the ordering reverses under attack.
5. **Tables 7 and 8** locate and characterise the failure: which class, and whether the model
   knew it was failing.
6. **Table 9** explains it. Sixty per cent of the missed hate posts are political criticism or
   arguably mislabelled, which sets a ceiling no architecture choice can lift.

That sequence is the argument. The numbers do not change, only the sentences between them.

**One thing to fix in the repository:** `true_hate_MISSED_categorised.csv` is not committed.
It is the record of the only annotation work in the project that required reading Hinglish,
and the artefact is separately assessed. It belongs in `writing/`.